# Pseudo-label 품질 검증 및 데이터셋 확장 전략

새 데이터셋에 AutoGaze 추론으로 생성한 pseudo-label이 NTP 학습에 쓸 만한지 검증하는 노트북입니다.

**다루는 내용**
1. 검증 개요 — greedy search vs self-distillation 이해
2. Pseudo-label 생성 — 새 데이터셋 비디오에 AutoGaze 추론 실행
3. Tier-1 통계 지표 — VideoMAE 없이 즉시 계산 가능한 프록시 메트릭
4. Tier-2 Reconstruction Quality — VideoMAE로 실제 복원 품질 비교
5. 원본 레이블과 분포 비교 (선택)
6. 판단 기준 & 전략 선택 (go / no-go checklist)

**사전 조건**
```bash
source .venv/bin/activate
pip install -e ".[dev]"
```

---
## 0. 환경 확인

In [ ]:
import sys, platform
import matplotlib
import matplotlib.font_manager as fm
import torch

# ── 한글 폰트 설정 ─────────────────────────────────────────────────
def _setup_korean_font():
    _sys = platform.system()
    if _sys == 'Darwin':
        matplotlib.rcParams['font.family'] = 'AppleGothic'
        _font = 'AppleGothic'
    elif _sys == 'Windows':
        matplotlib.rcParams['font.family'] = 'Malgun Gothic'
        _font = 'Malgun Gothic'
    else:
        _candidates = [
            f.name for f in fm.fontManager.ttflist
            if any(k in f.name for k in ('Nanum', 'Malgun', 'Gothic', 'Batang', 'Dotum'))
        ]
        if _candidates:
            matplotlib.rcParams['font.family'] = _candidates[0]
            _font = _candidates[0]
        else:
            print("⚠  한글 폰트 없음 — sudo apt-get install fonts-nanum && fc-cache -fv")
            return
    matplotlib.rcParams['axes.unicode_minus'] = False
    print(f"[폰트] {_font}  (axes.unicode_minus=False)")

_setup_korean_font()
# ─────────────────────────────────────────────────────────────────

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"Device : CUDA — {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("Device : MPS (Apple Silicon)")
else:
    print("Device : CPU")

import autogaze
print("autogaze 패키지 로드 성공 ✓")

In [ ]:
from pathlib import Path

# ── 경로 설정 ────────────────────────────────────────────────────
MODEL_PATH    = "nvidia/AutoGaze"             # HF ID 또는 로컬 경로
VIDEOMAE_PT   = "../weights/VideoMAE_AutoGaze/videomae.pt"  # VideoMAE 가중치

# 검증할 비디오들이 있는 디렉터리 (없으면 예제 파일 단일 사용)
NEW_VIDEO_DIR  = "../assets"                  # 새 데이터셋 디렉터리
VIDEO_EXT      = ".mp4"

# 원본 NTP 학습용 레이블 (있으면 비교, 없으면 스킵)
ORIG_LABEL_JSON = None  # 예: "../AutoGaze-Training-Data/gazing_labels.json"

OUTPUT_DIR = Path("../results/validation")
LABEL_JSON = OUTPUT_DIR / "pseudo_labels.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────

# 비디오 파일 목록 수집
video_dir = Path(NEW_VIDEO_DIR)
video_files = sorted(video_dir.rglob(f"*{VIDEO_EXT}"))
print(f"검증 대상 비디오: {len(video_files)}개")
for vf in video_files:
    print(f"  {vf}")

---
## 1. 검증 개요 — 왜 pseudo-label 검증이 필요한가

### 원본 gazing label 생성 방법 (논문/README 기준)

> *"The data is constructed by collecting raw videos from existing video datasets and **labeling gazing sequences for a subset of it using a greedy-search algorithm**."*  
> — `weights/AutoGaze/README.md`

원본 레이블은 **greedy search**로 만든 것입니다.  
각 비디오에 대해 VideoMAE reconstruction loss를 가장 효과적으로 낮추는 패치 조합을 탐욕적으로 탐색합니다.  
이 greedy search 코드는 오픈소스에 포함되어 있지 않습니다.

### 옵션 B (pseudo-label)의 본질적 한계

```
[원본 파이프라인]
greedy search (최적해) ──→ gazing_labels.json ──→ NTP 학습 ──→ AutoGaze
     (teacher)                  (GT labels)                     (student)

[옵션 B 파이프라인]
AutoGaze 추론 ──→ pseudo-labels ──→ NTP 학습 ──→ AutoGaze'
  (student)       (근사된 teacher)              (self-distillation)
```

AutoGaze 자체가 greedy search를 **근사**한 모델입니다.  
따라서 pseudo-label로 NTP 학습을 하면 오차가 누적되는 **self-distillation** 구조가 됩니다.

### 언제 pseudo-label이 충분한가

| 조건 | 판단 |
| --- | --- |
| 새 도메인이 원본 학습 데이터와 유사한 경우 | pseudo-label 품질이 높아 NTP 초기화로 충분 |
| 새 도메인이 크게 다른 경우 (의료·위성·공장 등) | pseudo-label 신뢰도 낮음, Stage 2 RL만 사용 권장 |
| 반복 학습 (여러 iteration) | 매 iteration마다 품질 저하 위험, 원본 레이블 필수 |

---
## 2. Pseudo-label 생성

AutoGaze를 새 데이터셋 비디오들에 실행해 `gazing_labels.json`을 만듭니다.

In [ ]:
from autogaze.models.autogaze import AutoGaze, AutoGazeImageProcessor
from autogaze.utils import get_device

device = get_device()
print(f"디바이스: {device}")

print(f"\n모델 로드 중: '{MODEL_PATH}' ...")
transform = AutoGazeImageProcessor.from_pretrained(MODEL_PATH)
model     = AutoGaze.from_pretrained(MODEL_PATH).to(device)
model.eval()

CHUNK_SIZE = model.config.max_num_frames
NUM_TOKENS = model.config.num_vision_tokens_each_frame
SCALES     = [int(s) for s in model.config.scales.split("+")]

print(f"모델 로드 완료 ✓  (max_num_frames={CHUNK_SIZE}, tokens/frame={NUM_TOKENS})")

In [ ]:
import json
import numpy as np
from autogaze.datasets.video_utils import (
    read_video_pyav, sample_frame_indices, process_video_frames,
    transform_video_for_pytorch,
)
import av
from autogaze.infer import save_json

# ── 생성 파라미터 ────────────────────────────────────────────────
GAZING_RATIO = 0.75    # 최대 패치 비율 (논문 NTP 학습은 0.1, 여기서는 넉넉하게)
TASK_LOSS_REQ = None   # 임계값 없이 비율 기반으로 생성
# ─────────────────────────────────────────────────────────────────

all_labels: dict = {}

for video_path in video_files:
    try:
        container = av.open(str(video_path))
        total_frames = container.streams.video[0].frames
        indices = sample_frame_indices(
            clip_len=CHUNK_SIZE, frame_sample_rate=1,
            seg_len=total_frames, random_sample_frame=False,
        )
        raw = read_video_pyav(container, indices)
        container.close()
        raw = process_video_frames(raw, CHUNK_SIZE)

        video_tensor = transform_video_for_pytorch(raw, transform)
        video_tensor = video_tensor[None].to(device)

        with torch.inference_mode():
            gaze_out = model(
                {"video": video_tensor},
                gazing_ratio=GAZING_RATIO,
                task_loss_requirement=TASK_LOSS_REQ,
            )

        save_json(gaze_out, video_path, all_labels)
        n_real = int((~gaze_out["if_padded_gazing"]).sum().item())
        print(f"  {video_path.name}: {n_real} / {CHUNK_SIZE * NUM_TOKENS} 패치 선택")

    except Exception as e:
        print(f"  [오류] {video_path.name}: {e}")

# JSON 저장
with open(LABEL_JSON, "w") as f:
    json.dump(all_labels, f, indent=2)

print(f"\npseudo-label 생성 완료: {LABEL_JSON}  ({len(all_labels)} 비디오)")

---
## 3. Tier-1 통계 지표 — 빠른 프록시 검증

VideoMAE 없이 계산 가능한 지표들입니다.

| 지표 | 의미 | 기준값 |
| --- | --- | --- |
| **patch_ratio** | 전체 패치 중 선택 비율 | gazing_ratio와 일치해야 함 |
| **temporal_coverage** | 적어도 1개 패치가 있는 프레임 비율 | > 0.9 권장 |
| **spatial_entropy** | 선택 패치 위치의 엔트로피 | 높을수록 다양한 위치 선택 |
| **informativeness_ratio** | 선택 패치 영역의 픽셀 분산 / 전체 평균 분산 | > 1.0 이면 정보량 높은 패치 우선 선택 |

In [ ]:
# ── 레이블 JSON 로드 ─────────────────────────────────────────────
with open(LABEL_JSON) as f:
    labels = json.load(f)

print(f"레이블 수: {len(labels)} 비디오")
print(f"JSON 키 예시: {list(labels.keys())[0]}")
print()

# 각 비디오의 기본 통계 수집
stats = []
for vid_key, entry in labels.items():
    gp = entry["gazing_pos"]      # [[pos,...], ...] — 프레임별 리스트
    n_frames     = len(gp)
    total_tokens = n_frames * NUM_TOKENS
    patch_counts = [len(frame_gp) for frame_gp in gp]
    n_selected   = sum(patch_counts)
    n_frames_covered = sum(1 for c in patch_counts if c > 0)

    stats.append({
        "key": vid_key,
        "n_frames": n_frames,
        "total_tokens": total_tokens,
        "n_selected": n_selected,
        "patch_ratio": n_selected / total_tokens if total_tokens > 0 else 0,
        "temporal_coverage": n_frames_covered / n_frames if n_frames > 0 else 0,
        "patch_counts": patch_counts,
    })

# 요약 출력
ratios   = [s["patch_ratio"] for s in stats]
coverage = [s["temporal_coverage"] for s in stats]

print(f"{'지표':25s} {'평균':>8} {'최소':>8} {'최대':>8}")
print("-" * 55)
print(f"{'patch_ratio':25s} {np.mean(ratios):8.3f} {np.min(ratios):8.3f} {np.max(ratios):8.3f}")
print(f"{'temporal_coverage':25s} {np.mean(coverage):8.3f} {np.min(coverage):8.3f} {np.max(coverage):8.3f}")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import entropy

# ── 공간 엔트로피 계산 ──────────────────────────────────────────
# 각 비디오의 patch position 히스토그램에서 엔트로피 계산
# (NUM_TOKENS 버킷: 어느 위치를 얼마나 자주 선택했는가)

spatial_entropies = []
max_entropy = np.log(NUM_TOKENS)   # 균등 분포일 때 최대값

for s in stats:
    all_pos = []
    for frame_gp in labels[s["key"]]["gazing_pos"]:
        # 각 position을 프레임-로컬 토큰 인덱스로 변환
        frame_idx_offset = labels[s["key"]]["gazing_pos"].index(frame_gp)
        local_pos = [p % NUM_TOKENS for p in frame_gp]
        all_pos.extend(local_pos)
    
    if len(all_pos) > 0:
        counts = np.bincount(all_pos, minlength=NUM_TOKENS).astype(float)
        counts += 1e-9  # smoothing
        ent = entropy(counts / counts.sum())  # nats
        spatial_entropies.append(ent / max_entropy)  # 0~1로 정규화
    else:
        spatial_entropies.append(0.0)

for s, ent in zip(stats, spatial_entropies):
    s["spatial_entropy_norm"] = ent

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. patch_ratio 분포
axes[0].hist(ratios, bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(GAZING_RATIO, color="tomato", linestyle="--", label=f"설정값 {GAZING_RATIO}")
axes[0].set_xlabel("patch_ratio")
axes[0].set_ylabel("비디오 수")
axes[0].set_title("패치 선택 비율 분포")
axes[0].legend()

# 2. temporal_coverage 분포
axes[1].hist(coverage, bins=10, color="seagreen", edgecolor="white")
axes[1].axvline(0.9, color="tomato", linestyle="--", label="권장 기준 0.9")
axes[1].set_xlabel("temporal_coverage")
axes[1].set_ylabel("비디오 수")
axes[1].set_title("시간적 커버리지 분포")
axes[1].legend()

# 3. spatial_entropy (정규화) 분포
axes[2].hist(spatial_entropies, bins=20, color="darkorange", edgecolor="white")
axes[2].axvline(0.7, color="tomato", linestyle="--", label="권장 기준 0.7")
axes[2].set_xlabel("spatial entropy (정규화, 0~1)")
axes[2].set_ylabel("비디오 수")
axes[2].set_title("공간 엔트로피 분포")
axes[2].legend()

plt.suptitle("Pseudo-label Tier-1 통계 지표", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\n공간 엔트로피 (정규화) 평균: {np.mean(spatial_entropies):.3f}")
print(f"  1.0 = 완전 균등, 0 = 한 위치에 집중")

In [ ]:
# ── informativeness_ratio 계산 ──────────────────────────────────
# 선택된 패치 영역의 픽셀 분산 vs 전체 패치 분산 평균
# 높을수록 AutoGaze가 정보량이 높은 영역을 우선 선택했음을 의미

from autogaze.datasets.video_utils import (
    read_video_pyav, sample_frame_indices, process_video_frames,
    transform_video_for_pytorch,
)

def compute_informativeness_ratio(video_path, label_entry, transform, scales=SCALES, num_tokens=NUM_TOKENS):
    """
    선택 패치 vs 전체 패치의 픽셀 분산 비율 계산.
    ratio > 1.0 → AutoGaze가 고분산(정보량 높은) 패치 우선 선택
    ratio ≈ 1.0 → 무작위 선택과 유사
    """
    try:
        container = av.open(str(video_path))
        total_frames = container.streams.video[0].frames
        indices = sample_frame_indices(
            clip_len=CHUNK_SIZE, frame_sample_rate=1,
            seg_len=total_frames, random_sample_frame=False,
        )
        raw = read_video_pyav(container, indices)
        container.close()
        raw = process_video_frames(raw, CHUNK_SIZE)
    except Exception:
        return None

    # 가장 큰 스케일 사용 (224px)
    scale = scales[-1]
    import torch.nn.functional as F
    import torch

    selected_vars, all_vars = [], []
    gazing_pos_frames = label_entry["gazing_pos"]
    n_frames = min(len(gazing_pos_frames), len(raw))

    for t in range(n_frames):
        frame = torch.from_numpy(raw[t]).permute(2, 0, 1).float() / 255.0  # (3, H, W)
        frame_r = F.interpolate(
            frame.unsqueeze(0), size=(scale, scale),
            mode="bicubic", align_corners=False,
        ).squeeze(0)  # (3, scale, scale)

        # 패치 크기 결정 (scale-224 → 16px 패치, 총 14×14 = 196개)
        n_patches_1d = int(num_tokens ** 0.5) if scale == scales[-1] else int((num_tokens // len(scales)) ** 0.5)
        # 간단히 scale-224의 14×14 grid 사용
        grid_size = 14
        patch_px  = scale // grid_size

        # 프레임 내 각 패치의 분산 계산
        patch_vars = []
        for pi in range(grid_size):
            for pj in range(grid_size):
                patch = frame_r[:, pi*patch_px:(pi+1)*patch_px, pj*patch_px:(pj+1)*patch_px]
                patch_vars.append(patch.var().item())
        patch_vars = np.array(patch_vars)  # (grid_size^2,)

        mean_all_var = patch_vars.mean()
        all_vars.append(mean_all_var)

        # 선택된 패치의 분산만 추출
        sel_pos = gazing_pos_frames[t]  # global positions
        if len(sel_pos) > 0:
            # local position = global_pos % num_tokens
            local_pos = [p % (grid_size * grid_size) for p in sel_pos]
            local_pos = [p for p in local_pos if p < len(patch_vars)]
            if local_pos:
                selected_vars.append(patch_vars[local_pos].mean())

    if not selected_vars or np.mean(all_vars) == 0:
        return 1.0

    return float(np.mean(selected_vars) / (np.mean(all_vars) + 1e-8))


# 비디오별 informativeness_ratio 계산
print("informativeness_ratio 계산 중 ...")
info_ratios = []
for vf in video_files:
    from autogaze.datasets.video_utils import get_relative_video_path
    rel_key = get_relative_video_path(str(vf))
    entry = labels.get(rel_key)
    if entry is None:
        # key 매칭 시도
        matching = [k for k in labels if vf.name in k or k in str(vf)]
        entry = labels[matching[0]] if matching else None
    if entry:
        r = compute_informativeness_ratio(vf, entry, transform)
        if r is not None:
            info_ratios.append(r)
            print(f"  {vf.name}: informativeness_ratio = {r:.3f}")

if info_ratios:
    print(f"\n평균 informativeness_ratio: {np.mean(info_ratios):.3f}")
    print(f"  > 1.0 → 정보량 높은 패치 우선 선택 (양호)")
    print(f"  ≈ 1.0 → 무작위와 유사 (레이블 품질 의심)")
else:
    print("⚠  계산 가능한 비디오 없음")

---
## 4. Tier-2 Reconstruction Quality — VideoMAE 기반 검증

**이 섹션은 VideoMAE 가중치(`weights/VideoMAE_AutoGaze/videomae.pt`)가 있어야 실행됩니다.**

핵심 비교:
- **pseudo-label reconstruction loss**: AutoGaze가 선택한 패치로 VideoMAE 복원
- **random baseline reconstruction loss**: 같은 수의 패치를 랜덤 선택해 복원
- **efficiency_gain** = (random_loss - pseudo_loss) / random_loss

> efficiency_gain > 0 → pseudo-label이 random보다 유의미한 패치 선택  
> efficiency_gain ≈ 0 → 품질 의심, Stage 2 RL only 전략 권장

In [ ]:
# VideoMAE 가중치 존재 여부 확인
videomae_pt = Path(VIDEOMAE_PT)
VIDEOMAE_AVAILABLE = videomae_pt.exists()

if VIDEOMAE_AVAILABLE:
    print(f"VideoMAE 가중치 발견: {videomae_pt}")
    print(f"파일 크기: {videomae_pt.stat().st_size / 1e9:.2f} GB")
else:
    print(f"⚠  VideoMAE 가중치 없음: {videomae_pt}")
    print("   Tier-2 검증을 건너뜁니다.")
    print()
    print("   가중치 다운로드:")
    print("   hf download bfshi/VideoMAE_AutoGaze --local-dir weights/VideoMAE_AutoGaze")

In [ ]:
if VIDEOMAE_AVAILABLE:
    from omegaconf import OmegaConf
    from autogaze.tasks.video_mae_reconstruction.modeling_video_mae import ViTMAEForPreTraining
    from transformers import VivitImageProcessor
    import yaml

    # VideoMAE 설정 로드
    cfg_path = videomae_pt.parent / "config.yaml"
    if cfg_path.exists():
        with open(cfg_path) as f:
            vmae_cfg = OmegaConf.create(yaml.safe_load(f))
        task_cfg = vmae_cfg.get("task", OmegaConf.create({}))
    else:
        task_cfg = OmegaConf.create({})

    RECON_MODEL = task_cfg.get("recon_model", "facebook/vit-mae-large")
    SCALES_STR  = task_cfg.get("scales", "32+64+112+224")

    print(f"VideoMAE 모델: {RECON_MODEL}")
    print(f"스케일: {SCALES_STR}")
    print("\nVideoMAE 로드 중 (시간이 걸릴 수 있습니다) ...")

    # 모델 아키텍처 초기화
    recon_model_config = OmegaConf.create({
        "loss_type": "l1",
        "loss_weights": "1",
    })
    mae = ViTMAEForPreTraining.from_pretrained(
        RECON_MODEL,
        attn_implementation="sdpa",
        scales=SCALES_STR,
        **OmegaConf.to_container(recon_model_config),
    )

    # 커스텀 가중치 로드
    state = torch.load(str(videomae_pt), map_location="cpu", weights_only=True)
    missing, unexpected = mae.load_state_dict(state, strict=False)
    print(f"가중치 로드 완료 ✓  (missing={len(missing)}, unexpected={len(unexpected)})")

    mae_transform = VivitImageProcessor.from_pretrained(RECON_MODEL, size=224)
    mae = mae.to(device).eval()
    print("VideoMAE 준비 완료 ✓")

In [ ]:
if not VIDEOMAE_AVAILABLE:
    print("⚠  VideoMAE 없음 — Tier-2 셀 건너뜁니다.")
else:
    # ── Pseudo-label vs Random reconstruction loss 비교 ─────────────
    from autogaze.datasets.collate import process_gazing_info
    import torch

    def build_gazing_info_from_label(label_entry, device):
        """JSON 레이블 엔트리를 trainer 입력 형식으로 변환."""
        gazing_pos = label_entry["gazing_pos"]
        task_losses = label_entry["task_losses"]
        info = process_gazing_info([gazing_pos], [task_losses])
        return {k: v.to(device) for k, v in info.items()}

    def random_gazing_info(gazing_info_ref, num_tokens=NUM_TOKENS, n_frames=CHUNK_SIZE, device="cpu"):
        """같은 패치 수를 랜덤 위치에서 선택한 gazing_info 생성."""
        num_each = gazing_info_ref["num_gazing_each_frame"]  # (T,)
        n_frames = len(num_each)
        rand_pos_frames, rand_loss_frames = [], []

        for t in range(n_frames):
            cnt = int(num_each[t].item()) - 1  # -1 for EOS
            cnt = max(0, cnt)
            rand_pos = torch.randperm(num_tokens)[:cnt].tolist()
            global_pos = [p + t * num_tokens for p in rand_pos]
            rand_pos_frames.append(global_pos)
            rand_loss_frames.append([0.5] * cnt)

        info = process_gazing_info([rand_pos_frames], [rand_loss_frames])
        return {k: v.to(device) for k, v in info.items()}

    results = []
    print("Reconstruction loss 비교 중 ...\n")

    for vf in video_files:
        from autogaze.datasets.video_utils import get_relative_video_path
        rel_key = get_relative_video_path(str(vf))
        entry = labels.get(rel_key)
        if entry is None:
            matching = [k for k in labels if vf.name in k or k in str(vf)]
            entry = labels[matching[0]] if matching else None
        if entry is None:
            continue

        try:
            container = av.open(str(vf))
            total_frames = container.streams.video[0].frames
            indices = sample_frame_indices(
                clip_len=CHUNK_SIZE, frame_sample_rate=1,
                seg_len=total_frames, random_sample_frame=False,
            )
            raw = read_video_pyav(container, indices)
            container.close()
            raw = process_video_frames(raw, CHUNK_SIZE)

            # VideoMAE용 video tensor
            video_task = transform_video_for_pytorch(raw, mae_transform)
            video_task = video_task[None].to(device)

            # Pseudo-label gazing_info
            g_pseudo = build_gazing_info_from_label(entry, device)

            # Random gazing_info (같은 패치 수)
            g_random = random_gazing_info(g_pseudo, num_tokens=NUM_TOKENS, device=device)

            frame_idx = torch.tensor([0], device=device)  # 첫 프레임만 복원

            with torch.inference_mode():
                out_pseudo = mae(video_task, gazing_info=g_pseudo,
                                 frame_idx_to_reconstruct=frame_idx,
                                 interpolate_pos_encoding=True)
                out_random = mae(video_task, gazing_info=g_random,
                                 frame_idx_to_reconstruct=frame_idx,
                                 interpolate_pos_encoding=True)

            loss_pseudo = float(out_pseudo.loss_mean.item())
            loss_random = float(out_random.loss_mean.item())
            gain = (loss_random - loss_pseudo) / (loss_random + 1e-8)

            results.append({
                "video": vf.name,
                "loss_pseudo": loss_pseudo,
                "loss_random": loss_random,
                "efficiency_gain": gain,
            })
            print(f"  {vf.name}")
            print(f"    pseudo-label loss: {loss_pseudo:.4f}")
            print(f"    random baseline:   {loss_random:.4f}")
            print(f"    efficiency_gain:   {gain:+.3f}  {'(양호)' if gain > 0.05 else '(미미함)'}")

        except Exception as e:
            print(f"  [오류] {vf.name}: {e}")

In [ ]:
if VIDEOMAE_AVAILABLE and results:
    import matplotlib.pyplot as plt
    import numpy as np

    videos    = [r["video"] for r in results]
    l_pseudo  = [r["loss_pseudo"] for r in results]
    l_random  = [r["loss_random"] for r in results]
    gains     = [r["efficiency_gain"] for r in results]

    x = np.arange(len(videos))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss 비교 막대 그래프
    axes[0].bar(x - width/2, l_pseudo, width, label="Pseudo-label", color="steelblue")
    axes[0].bar(x + width/2, l_random, width, label="Random baseline", color="tomato", alpha=0.7)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(videos, rotation=20, ha="right", fontsize=9)
    axes[0].set_ylabel("Reconstruction Loss")
    axes[0].set_title("Pseudo-label vs Random Reconstruction Loss")
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    # Efficiency gain
    colors = ["seagreen" if g > 0.05 else "tomato" for g in gains]
    bars = axes[1].bar(x, gains, color=colors, edgecolor="white")
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].axhline(0.05, color="gray", linestyle="--", alpha=0.5, label="권장 기준 0.05")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(videos, rotation=20, ha="right", fontsize=9)
    axes[1].set_ylabel("Efficiency Gain")
    axes[1].set_title("Reconstruction Efficiency Gain\n(pseudo - random) / random")
    for bar, g in zip(bars, gains):
        axes[1].text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{g:+.3f}", ha="center", fontsize=9,
        )
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    plt.suptitle("Tier-2: VideoMAE Reconstruction Quality 비교", fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"\n평균 efficiency_gain: {np.mean(gains):.3f}")
    print(f"양호 비디오 (gain > 0.05): {sum(g > 0.05 for g in gains)} / {len(gains)}")

---
## 5. 원본 레이블과 분포 비교 (선택)

`ORIG_LABEL_JSON`이 설정된 경우, 원본 gazing_labels.json과 pseudo-label의 분포를 비교합니다.

비교 대상:
- 프레임별 패치 수 분포
- 공간 엔트로피
- Position 빈도 맵 (어떤 위치를 많이 선택하는가)

In [ ]:
if ORIG_LABEL_JSON is None:
    print("ORIG_LABEL_JSON이 설정되지 않았습니다. 이 섹션을 건너뜁니다.")
    print()
    print("원본 gazing_labels.json이 있으면 아래처럼 설정:")
    print('  ORIG_LABEL_JSON = "../AutoGaze-Training-Data/gazing_labels.json"')
else:
    with open(ORIG_LABEL_JSON) as f:
        orig_labels = json.load(f)

    print(f"원본 레이블: {len(orig_labels)} 비디오")
    print(f"Pseudo 레이블: {len(labels)} 비디오")

    # 프레임별 패치 수 분포 비교
    def get_patch_count_distribution(label_dict, n_sample=500):
        counts = []
        for entry in list(label_dict.values())[:n_sample]:
            for frame_gp in entry["gazing_pos"]:
                counts.append(len(frame_gp))
        return np.array(counts)

    orig_counts  = get_patch_count_distribution(orig_labels)
    pseudo_counts = get_patch_count_distribution(labels)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 히스토그램 비교
    bins = np.linspace(0, max(orig_counts.max(), pseudo_counts.max()), 40)
    axes[0].hist(orig_counts, bins=bins, alpha=0.6, label="원본 레이블", color="steelblue", density=True)
    axes[0].hist(pseudo_counts, bins=bins, alpha=0.6, label="Pseudo-label", color="tomato", density=True)
    axes[0].set_xlabel("프레임당 패치 수")
    axes[0].set_ylabel("밀도")
    axes[0].set_title("프레임당 패치 수 분포 비교")
    axes[0].legend()

    # Position 빈도 맵 비교
    def get_position_heatmap(label_dict, num_tokens=NUM_TOKENS, grid=14, n_sample=200):
        freq = np.zeros((grid, grid))
        count = 0
        for entry in list(label_dict.values())[:n_sample]:
            for frame_gp in entry["gazing_pos"]:
                for p in frame_gp:
                    local = p % (grid * grid)
                    if local < grid * grid:
                        freq[local // grid, local % grid] += 1
                        count += 1
        return freq / (count + 1e-8)

    orig_freq   = get_position_heatmap(orig_labels)
    pseudo_freq = get_position_heatmap(labels)
    diff_map    = pseudo_freq - orig_freq

    vmax = max(abs(diff_map).max(), 1e-6)
    im = axes[1].imshow(diff_map, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    plt.colorbar(im, ax=axes[1])
    axes[1].set_title("Position 빈도 차이 맵\n(Pseudo - 원본, 14×14 grid)")
    axes[1].set_xlabel("가로 위치")
    axes[1].set_ylabel("세로 위치")

    plt.suptitle("원본 레이블 vs Pseudo-label 분포 비교", fontsize=13)
    plt.tight_layout()
    plt.show()

    # KL divergence로 분포 유사도 측정
    from scipy.stats import entropy
    hist_bins = np.arange(0, max(orig_counts.max(), pseudo_counts.max()) + 2)
    orig_dist, _  = np.histogram(orig_counts,   bins=hist_bins, density=True)
    pseudo_dist,_ = np.histogram(pseudo_counts, bins=hist_bins, density=True)
    orig_dist  = orig_dist  + 1e-9
    pseudo_dist= pseudo_dist+ 1e-9
    kl = float(entropy(pseudo_dist / pseudo_dist.sum(), orig_dist / orig_dist.sum()))
    print(f"\nKL divergence (Pseudo ‖ 원본): {kl:.4f}")
    print(f"  KL < 0.1  → 분포 매우 유사 (NTP 학습 적합)")
    print(f"  KL < 0.3  → 허용 범위")
    print(f"  KL >= 0.3 → 분포 차이 큼 — Stage 2 RL only 고려")

---
## 6. 판단 기준 & 전략 선택

지금까지 계산한 지표를 종합해 학습 전략을 결정합니다.

In [ ]:
# ── 판단 기준 (임계값 수정 가능) ────────────────────────────────
THRESHOLD_TEMPORAL_COVERAGE  = 0.90   # 시간적 커버리지 하한
THRESHOLD_SPATIAL_ENTROPY    = 0.70   # 공간 엔트로피 하한 (정규화)
THRESHOLD_INFORMATIVENESS    = 1.05   # informativeness_ratio 하한
THRESHOLD_EFFICIENCY_GAIN    = 0.05   # Tier-2 efficiency gain 하한
# ─────────────────────────────────────────────────────────────────

checks = {}

# Tier-1 체크
mean_coverage = np.mean(coverage) if coverage else 0
mean_entropy  = np.mean(spatial_entropies) if spatial_entropies else 0
mean_info     = np.mean(info_ratios) if info_ratios else 1.0

checks["시간적 커버리지 >= 0.90"] = (mean_coverage >= THRESHOLD_TEMPORAL_COVERAGE, mean_coverage)
checks["공간 엔트로피 >= 0.70  "] = (mean_entropy  >= THRESHOLD_SPATIAL_ENTROPY,  mean_entropy)
checks["informativeness > 1.05"] = (mean_info     >= THRESHOLD_INFORMATIVENESS,  mean_info)

# Tier-2 체크 (있는 경우)
if VIDEOMAE_AVAILABLE and results:
    mean_gain = np.mean(gains)
    checks["efficiency_gain >= 0.05  "] = (mean_gain >= THRESHOLD_EFFICIENCY_GAIN, mean_gain)

# KL divergence 체크 (있는 경우)
if ORIG_LABEL_JSON and "kl" in dir():
    checks["KL divergence < 0.30   "] = (kl < 0.30, kl)

# 결과 출력
print("=" * 55)
print("Pseudo-label 품질 검증 결과")
print("=" * 55)
passed = 0
for check_name, (ok, value) in checks.items():
    status = "PASS" if ok else "FAIL"
    symbol = "✓" if ok else "✗"
    print(f"  [{status}] {symbol} {check_name}: {value:.4f}")
    if ok:
        passed += 1

n_checks = len(checks)
print("-" * 55)
print(f"  통과: {passed} / {n_checks}")
print("=" * 55)

In [ ]:
# ── 전략 권장 ────────────────────────────────────────────────────
n_tier1 = sum(1 for k in checks if "KL" not in k and "efficiency" not in k)
tier1_passed = sum(v[0] for k, v in checks.items() if "KL" not in k and "efficiency" not in k)

print("\n권장 전략")
print("─" * 55)

if tier1_passed == n_tier1:
    print("  전략 A: Stage 1 NTP + Stage 2 RL (권장)")
    print()
    print("  이유: 모든 Tier-1 기준 통과.")
    print("  pseudo-label이 충분한 품질로 NTP 초기화에 사용 가능.")
    print()
    print("  실행:")
    print("    1. LABEL_JSON을 NTP dataset.gt_gazing_pos_paths.train에 지정")
    print("    2. 02_train_ntp_ko.ipynb 참고해 Stage 1 학습")
    print("    3. 03_train_rl_ko.ipynb 참고해 Stage 2 RL 학습")
elif tier1_passed >= n_tier1 // 2:
    print("  전략 B: Stage 2 RL only (안전한 선택)")
    print()
    print("  이유: Tier-1 기준 일부 미통과.")
    print("  NTP pseudo-label의 품질이 불확실하므로")
    print("  pretrained nvidia/AutoGaze에서 바로 RL fine-tuning.")
    print()
    print("  실행:")
    print("    - gt_gazing_pos_paths: 비워두기 (label 불필요)")
    print("    - 03_train_rl_ko.ipynb 참고해 Stage 2 RL 학습")
    print("    - gaze_weights=nvidia/AutoGaze로 초기화")
else:
    print("  전략 C: 데이터 재검토 필요")
    print()
    print("  이유: 대부분의 품질 기준 미통과.")
    print("  가능한 원인:")
    print("    - 비디오 해상도/포맷이 학습 분포와 크게 다름")
    print("    - 비디오 길이가 너무 짧거나 내용이 거의 없음 (정적 화면)")
    print("    - AutoGaze 도메인 외 콘텐츠 (의료·위성 등)")
    print()
    print("  권장 조치:")
    print("    1. 비디오 샘플을 직접 확인 (01_autogaze_tutorial_ko.ipynb)")
    print("    2. 해상도/포맷 전처리 후 재시도")
    print("    3. 도메인 차이가 크면 greedy search label 직접 생성 고려")

print("─" * 55)

In [ ]:
# ── 데이터셋 확장 체크리스트 요약 ────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

check_names = list(checks.keys())
check_vals  = [v[1] for v in checks.values()]
check_ok    = [v[0] for v in checks.values()]

fig, ax = plt.subplots(figsize=(10, max(3, 0.6 * len(check_names))))
y_pos = np.arange(len(check_names))

colors = ["seagreen" if ok else "tomato" for ok in check_ok]
bars = ax.barh(y_pos, check_vals, color=colors, edgecolor="white", height=0.5)

# 임계선
thresholds = {
    "시간적 커버리지 >= 0.90": THRESHOLD_TEMPORAL_COVERAGE,
    "공간 엔트로피 >= 0.70  ": THRESHOLD_SPATIAL_ENTROPY,
    "informativeness > 1.05": THRESHOLD_INFORMATIVENESS,
    "efficiency_gain >= 0.05  ": THRESHOLD_EFFICIENCY_GAIN,
    "KL divergence < 0.30   ": 0.30,
}
for yi, name in enumerate(check_names):
    thr = thresholds.get(name)
    if thr:
        ax.axvline(x=thr, ymin=(yi - 0.3) / len(check_names),
                   ymax=(yi + 0.3 + 1) / len(check_names),
                   color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.text(max(check_vals) * 0.02, yi, f"{check_vals[yi]:.3f}",
            va="center", ha="left", fontsize=10, color="white", fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(check_names, fontsize=10)
ax.set_xlabel("측정값")
ax.set_title("Pseudo-label 품질 검증 체크리스트")
ax.invert_yaxis()

pass_patch = mpatches.Patch(color="seagreen", label="PASS")
fail_patch = mpatches.Patch(color="tomato",   label="FAIL")
ax.legend(handles=[pass_patch, fail_patch], loc="lower right")
plt.tight_layout()
plt.show()

---
## 정리

### 검증 지표 요약

| 지표 | 계층 | 필요 도구 | 해석 |
| --- | --- | --- | --- |
| temporal_coverage | Tier-1 | 없음 | 모든 프레임이 커버되는지 |
| spatial_entropy | Tier-1 | 없음 | 공간적으로 다양한 위치를 선택하는지 |
| informativeness_ratio | Tier-1 | 없음 | 고분산(중요) 패치를 우선 선택하는지 |
| efficiency_gain | Tier-2 | VideoMAE | 실제 복원 품질이 랜덤보다 높은지 |
| KL divergence | 선택 | 원본 레이블 | 원본 레이블 분포와 얼마나 유사한지 |

### 전략 결정 트리

```
모든 Tier-1 통과?
├─ YES → Stage 1 NTP(pseudo-label) + Stage 2 RL
│        (02_train_ntp_ko → 03_train_rl_ko)
│
└─ NO  → Tier-1 절반 이상 통과?
         ├─ YES → Stage 2 RL only
         │        (nvidia/AutoGaze → 03_train_rl_ko)
         │
         └─ NO  → 데이터 재검토
                  (해상도·포맷 전처리 또는 greedy search label 생성)
```

### 다음 단계
- `02_train_ntp_ko.ipynb` — Stage 1 NTP 학습 (pseudo-label 사용 시)
- `03_train_rl_ko.ipynb` — Stage 2 GRPO RL 학습
- `GUIDE_KO.md` 섹션 5 — 멀티-GPU 학습 설정